In [58]:
# Schema relationships:
# Games (1) --- (Many) Recommendations   [joined on app_id]
# Users (1) --- (Many) Recommendations   [joined on user_id]

In [59]:
import pandas as pd

#chunking randomly from the dataset to ensure distribution of data is fair

chunks = []
chunk_size = 500_000
sample_fraction = 0.02

for chunk in pd.read_csv("data/recommendations.csv",chunksize=chunk_size):
    sampled = chunk.sample(frac=sample_fraction,random_state=0)
    chunks.append(sampled)

recommendations_df = pd.concat(chunks,ignore_index=True)

#finding all unique ids - and only taking those user_ids from user dataset chunks to ensure no join conflict
unique_ids = set(recommendations_df["user_id"].unique())
print(len(recommendations_df))

823096


In [60]:
chunks = []
chunk_size = 500_000
sample_fraction = 0.025

for chunk in pd.read_csv("data/users.csv",chunksize=chunk_size):
    sampled = chunk[chunk["user_id"].isin(unique_ids)]
    chunks.append(sampled)
    
users_df = pd.concat(chunks,ignore_index=True)
print(len(users_df))

731788


In [61]:
games_df = pd.read_csv("data/games.csv")
print(len(games_df))

50872


In [62]:
import sqlite3

#connecting to database - enabling foreign keys - creating new table skeletons with datatypes and pk-fk relations mentioned

connection = sqlite3.connect("steam.db")
connection.execute("PRAGMA foreign_keys=ON")

connection.execute("DROP TABLE IF EXISTS recommendations")
connection.execute("DROP TABLE IF EXISTS users")
connection.execute("DROP TABLE IF EXISTS games")

connection.execute("Create table Users(user_id INTEGER NOT NULL PRIMARY KEY, products INTEGER NOT NULL, reviews INTEGER NOT NULL)")

connection.execute("Create table Games(app_id INTEGER NOT NULL PRIMARY KEY, title VARCHAR(200) NOT NULL, date_release DATE NOT NULL, win BOOLEAN NOT NULL, mac BOOLEAN NOT NULL, linux BOOLEAN NOT NULL, rating VARCHAR(100) NOT NULL, positive_ratio INTEGER NOT NULL, user_reviews INTEGER NOT NULL, price_final DOUBLE NOT NULL, price_original DOUBLE NOT NULL, discount DOUBLE NOT NULL, steam_deck BOOLEAN NOT NULL)")


connection.execute("Create table Recommendations (app_id INTEGER NOT NULL, helpful INTEGER NOT NULL, funny INTEGER NOT NULL, date DATE NOT NULL, is_recommended BOOLEAN NOT NULL, hours DOUBLE NOT NULL, user_id INTEGER NOT NULL, review_id INTEGER NOT NULL PRIMARY KEY, CONSTRAINT fk_Users FOREIGN KEY (user_id) REFERENCES Users(user_id), CONSTRAINT fk_Games FOREIGN KEY(app_id) REFERENCES Games(app_id))")

connection.commit()


In [63]:
#filling the skeleton with the dataframe data

users_df.to_sql("Users",connection,if_exists="append",index=False)
games_df.to_sql("Games",connection,if_exists="append",index=False)
recommendations_df.to_sql("Recommendations",connection,if_exists="append",index=False)

823096

In [64]:
import os
print(f"{os.path.getsize('steam.db') / (1024*1024):.1f} MB")

49.2 MB


In [65]:
pd.read_sql_query("select * from Recommendations limit 15;",connection)


,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,431960,0,0,2021-09-02,1,934.1,1166406,61
1,291550,0,0,2022-04-18,1,124.9,3203259,117
2,1332010,0,0,2022-07-19,1,5.7,7716904,168
3,1172620,0,0,2021-12-30,1,385.7,1586796,244
4,250900,0,0,2021-03-28,1,594.3,5884941,270
5,440900,0,0,2020-07-01,1,269.9,7291153,284
6,266410,0,0,2015-10-30,1,157.0,7654118,442
7,311210,2,0,2016-02-26,0,33.8,3515107,524
8,431960,0,0,2018-11-16,1,28.4,11673080,713
9,239030,2,0,2016-02-12,1,21.8,11793999,715


In [66]:
pd.read_sql_query("select * from Users limit 15;",connection)


,user_id,products,reviews
0,19,54,2
1,20,57,1
2,66,4,1
3,89,105,5
4,119,236,6
5,131,75,1
6,133,17,3
7,162,55,1
8,205,35,4
9,225,96,4


In [67]:
pd.read_sql_query("select * from Games limit 15;",connection)

,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,discount,steam_deck
0,10,Counter-Strike,2000-11-01,1,1,1,Overwhelmingly Positive,97,136118,9.99,9.99,0.0,1
1,20,Team Fortress Classic,1999-04-01,1,1,1,Very Positive,85,5146,4.99,4.99,0.0,1
2,30,Day of Defeat,2003-05-01,1,1,1,Very Positive,87,3489,4.99,4.99,0.0,1
3,40,Deathmatch Classic,2001-06-01,1,1,1,Mostly Positive,79,1757,4.99,4.99,0.0,1
4,50,Half-Life: Opposing Force,1999-11-01,1,1,1,Overwhelmingly Positive,95,14057,4.99,4.99,0.0,1
5,60,Ricochet,2000-11-01,1,1,1,Very Positive,80,3395,4.99,4.99,0.0,1
6,70,Half-Life,1998-11-08,1,1,1,Overwhelmingly Positive,96,72475,9.99,9.99,0.0,1
7,80,Counter-Strike: Condition Zero,2004-03-01,1,1,1,Very Positive,90,16187,9.99,9.99,0.0,1
8,130,Half-Life: Blue Shift,2001-06-01,1,1,1,Very Positive,91,10522,4.99,4.99,0.0,1
9,220,Half-Life 2,2007-10-10,1,1,1,Overwhelmingly Positive,97,122230,9.99,9.99,0.0,1
